# 01: Exploratory Data Analysis

What does the Rossmann data look like? Shapes, missingness, time coverage, and the
dominant sales patterns (weekly rhythm, promotions, store types). Transformation
logic lives in `src/rossmann`. This notebook only loads and visualises.

In [1]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '../src')
import matplotlib
matplotlib.use('Agg')
import pandas as pd
import matplotlib.pyplot as plt
from rossmann.data import loader

data = loader.load_raw(with_test=False)
df = data.train
print(f'Rows: {len(df):,} | Stores: {df["Store"].nunique()} | Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
df.head(3)

Rows: 1,017,209 | Stores: 1115 | Date range: 2013-01-01 to 2015-07-31


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"


In [2]:
# Missingness summary
df[['CompetitionDistance','CompetitionOpenSinceYear','Promo2SinceYear','PromoInterval']].isna().mean().rename('null_fraction').to_frame()

,null_fraction
CompetitionDistance,0.002597
CompetitionOpenSinceYear,0.317878
Promo2SinceYear,0.499436
PromoInterval,0.499436


In [3]:
# Sales distribution: raw vs log1p. The right skew motivates log-target training.
import numpy as np
open_sales = df.loc[df['Open']==1, 'Sales']
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
open_sales.hist(bins=60, ax=axes[0])
axes[0].set_title('Sales (open days)')
axes[0].set_xlabel('Sales')
np.log1p(open_sales).hist(bins=60, ax=axes[1])
axes[1].set_title('log1p(Sales)')
axes[1].set_xlabel('log1p(Sales)')
plt.tight_layout()
plt.savefig('../outputs/figures/eda_sales_dist.png', dpi=110)
plt.show()

In [4]:
# Two strongest short-term drivers: day-of-week rhythm and promotional uplift.
o = df[df['Open']==1]
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
dow_mean = o.groupby('DayOfWeek')['Sales'].mean()
dow_mean.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Mean sales by day of week (1=Mon, 7=Sun)')
axes[0].set_xlabel('')
promo_mean = o.groupby('Promo')['Sales'].mean()
promo_mean.plot(kind='bar', ax=axes[1], color=['#aaa', 'steelblue'])
axes[1].set_title('Mean sales: no promo vs promo')
axes[1].set_xticklabels(['No promo', 'Promo'], rotation=0)
plt.tight_layout()
plt.savefig('../outputs/figures/eda_dow_promo.png', dpi=110)
plt.show()
uplift = (promo_mean[1] / promo_mean[0] - 1) * 100
print(f'Promotion uplift: +{uplift:.1f}%')

Promotion uplift: +38.8%


In [5]:
# Sales profile by store type.
o.groupby('StoreType')['Sales'].agg(['mean','median','count']).round(0)

,mean,median,count
StoreType,,,
a,6925.0,6285.0,457077
b,10231.0,9130.0,15563
c,6933.0,6407.0,112978
d,6822.0,6395.0,258774


In [6]:
# Chain-level monthly trend: seasonality + year-over-year growth.
monthly = o.set_index('Date').resample('M')['Sales'].mean()
fig, ax = plt.subplots(figsize=(11, 3))
monthly.plot(ax=ax, color='steelblue')
ax.set_title('Mean daily sales by month (chain-wide)')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('../outputs/figures/eda_trend.png', dpi=110)
plt.show()